# Week 6 Exercise — Price-from-Description Predictor

**"The Price is Right"** — Predict a product's price in USD from its text description using a frontier LLM (zero-shot). Includes a small in-notebook eval set and a **Gradio** UI.

Set `OPENAI_API_KEY` in `.env`. Optional: run Ollama locally for the Ollama model.

In [1]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Add OPENAI_API_KEY to .env")
else:
    print("OpenAI API key loaded.")

# OpenAI models use the default https://api.openai.com/v1 base URL.
# Optional Ollama: run `ollama serve` and `ollama pull llama3.2` locally.
MODELS = {
    "OpenAI: gpt-4o-mini": {"backend": "openai", "model": "gpt-4o-mini"},
    "OpenAI: gpt-4o": {"backend": "openai", "model": "gpt-4o"},
    "Ollama: llama3.2": {
        "backend": "ollama",
        "model": "llama3.2",
        "base_url": os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
    },
}


def get_client(model_key):
    cfg = MODELS.get(model_key, next(iter(MODELS.values())))
    if cfg["backend"] == "openai":
        if not api_key:
            raise ValueError("Set OPENAI_API_KEY in .env for OpenAI models.")
        return OpenAI(api_key=api_key), cfg["model"]
    base = cfg.get("base_url", "http://localhost:11434/v1")
    return OpenAI(api_key="ollama", base_url=base), cfg["model"]

OpenAI API key loaded.


## Eval set

Small list of `(description, actual_price)` for computing MAE. Self-contained — no HuggingFace or course pricer modules.

In [2]:
EVAL_EXAMPLES = [
    ("Wireless Bluetooth earbuds, 24hr battery, noise cancelling", 49.99),
    ("Stainless steel water bottle 32oz insulated", 29.99),
    ("Mechanical keyboard RGB backlit, cherry MX switches", 129.00),
    ("USB-C laptop docking station dual 4K HDMI", 189.99),
    ("Portable power bank 20000mAh PD fast charge", 45.00),
    ("Standing desk mat anti-fatigue", 39.95),
    ("4K webcam with ring light and microphone", 79.99),
    ("Ergonomic office chair mesh back adjustable", 249.00),
    ("Smart thermostat WiFi programmable", 89.00),
    ("Noise cancelling over-ear headphones Bluetooth 5.0", 199.00),
]

print(f"Eval set: {len(EVAL_EXAMPLES)} (description, price) pairs")

Eval set: 10 (description, price) pairs


## Predictor

Prompt the LLM to estimate price in USD; respond with the number only. Parse the reply to extract a float.

In [3]:
PROMPT_TEMPLATE = """Estimate the price of this product in USD. Respond with only the price number, no explanation.

{description}"""


def parse_price(text):
    if not text:
        return 0.0
    text = text.strip().replace(",", "")
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    if match:
        try:
            return max(0.0, float(match.group(1)))
        except ValueError:
            pass
    return 0.0


def predict_price(description, model_key):
    if not description or not description.strip():
        return 0.0
    client, model_id = get_client(model_key)
    prompt = PROMPT_TEMPLATE.format(description=description.strip())
    try:
        r = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=15,
        )
        content = (r.choices[0].message.content or "").strip()
        return round(parse_price(content), 2)
    except Exception as e:
        return f"Error: {e}"


def evaluate_mae(model_key):
    errors = []
    for desc, actual in EVAL_EXAMPLES:
        pred = predict_price(desc, model_key)
        if isinstance(pred, (int, float)):
            errors.append(abs(pred - actual))
    if not errors:
        return "Could not compute (all predictions failed or errored)."
    mae = sum(errors) / len(errors)
    return f"MAE over {len(errors)} examples: ${mae:.2f}"


if api_key:
    print(evaluate_mae(list(MODELS.keys())[0]))
else:
    print("Skip MAE sample until OPENAI_API_KEY is set.")

MAE over 10 examples: $44.70


## Gradio UI

Enter a product description and get a predicted price. Optionally run evaluation to see MAE on the in-notebook set.

In [4]:
def run_eval(model_key):
    return evaluate_mae(model_key)


with gr.Blocks(title="Week 6 — Price from Description") as demo:
    gr.Markdown("## The Price is Right — Predict product price from description")
    with gr.Row():
        inp = gr.Textbox(
            label="Product description",
            lines=3,
            placeholder="e.g. Wireless Bluetooth earbuds, 24hr battery...",
        )
        out = gr.Number(label="Predicted price ($)")
    with gr.Row():
        model_dd = gr.Dropdown(
            choices=list(MODELS.keys()),
            value=list(MODELS.keys())[0],
            label="Model",
        )
        btn = gr.Button("Predict")
    btn.click(fn=lambda d, m: predict_price(d, m), inputs=[inp, model_dd], outputs=out)
    gr.Markdown("### Evaluation on in-notebook set")
    eval_btn = gr.Button("Run MAE evaluation")
    eval_out = gr.Textbox(label="MAE")
    eval_btn.click(fn=run_eval, inputs=[model_dd], outputs=eval_out)

demo.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
